# Integrated Workflow: Power + Fatigue + Optimization

This notebook demonstrates the complete workflow:
1. **Power Analysis** → Determine required completes
2. **Fatigue Audit** → Estimate drop-off rate
3. **Calculate Invitations** → Account for drop-off
4. **Optimize** → Reduce invitations by improving survey design

## Setup

In [ ]:
import sys
sys.path.append('..')

from src.power.calculators.means import MeansPowerCalc
from src.power.calculators.segments import SegmentPowerCalc
from src.power.integrations.dropoff import (
    calculate_invitations,
    estimate_dropoff_from_score,
    integrated_sample_plan
)
from src.power.integrations.optimizer import optimize_survey_design

import pandas as pd
import numpy as np

---

## Step 1: Power Analysis

**Research Question:** Do satisfaction scores differ between Treatment A and Control?

**Parameters:**
- Medium effect size (d = 0.5)
- α = 0.05, power = 0.80

In [ ]:
# Run power analysis
calc = MeansPowerCalc(test_type='independent')

power_result = calc.calculate(
    effect_size=0.5,
    alpha=0.05,
    power=0.80,
    solve_for='n'
)

required_completes = power_result['total_n']

print(f"✅ Power Analysis Complete")
print(f"Required completes: {required_completes}")
print(f"  ({power_result['n_per_group']} per group)")

---

## Step 2: Fatigue Audit

Suppose your survey has been audited and received a fatigue score of **68/100** (HIGH risk).

This indicates:
- Likely has loops or complex grids
- Expected drop-off: ~35%
- Will need significantly more invitations

In [ ]:
# Survey fatigue score from audit
fatigue_score = 68

# Estimate drop-off
dropoff_rate = estimate_dropoff_from_score(fatigue_score)

print(f"Fatigue Score: {fatigue_score}/100 (HIGH RISK)")
print(f"Estimated drop-off rate: {dropoff_rate:.1%}")

---

## Step 3: Calculate Invitations Needed

In [ ]:
# Calculate invitations accounting for drop-off
invitation_plan = calculate_invitations(
    required_completes=required_completes,
    dropoff_rate=dropoff_rate,
    response_rate=1.0,  # Assume everyone who gets invited starts
    buffer=0.0  # No safety buffer
)

print(f"📊 Sample Plan:")
print(f"Required completes: {invitation_plan['required_completes']}")
print(f"Drop-off rate: {invitation_plan['dropoff_rate']:.1%}")
print(f"\n🚨 Invitations needed: {invitation_plan['invitations_needed']}")
print(f"\nOverhead: {(invitation_plan['invitations_needed'] - required_completes) / required_completes:.1%}")
print(f"  (You need {invitation_plan['invitations_needed'] - required_completes} extra invitations to account for drop-off)")

---

## Step 4: Integrated Sample Plan

Combine power analysis and fatigue audit into a single plan:

In [ ]:
# Generate integrated plan
sample_plan = integrated_sample_plan(
    power_result=power_result,
    fatigue_score=fatigue_score,
    response_rate=1.0
)

print("=" * 60)
print("INTEGRATED SAMPLE PLAN")
print("=" * 60)

print(f"\n📈 Power Analysis:")
print(f"  Required completes: {sample_plan['power_analysis']['required_completes']}")
print(f"  Power: {sample_plan['power_analysis']['power']:.1%}")
print(f"  Effect size: {sample_plan['power_analysis']['effect_size']}")

print(f"\n📉 Fatigue Audit:")
print(f"  Fatigue score: {sample_plan['fatigue_audit']['fatigue_score']:.0f}/100")
print(f"  Risk level: {sample_plan['fatigue_audit']['risk_level']}")
print(f"  Estimated drop-off: {sample_plan['fatigue_audit']['estimated_dropoff_rate']:.1%}")

print(f"\n📧 Invitations:")
print(f"  Invitations to send: {sample_plan['sample_plan']['invitations_needed']}")
print(f"  Expected completes: {sample_plan['sample_plan']['expected_completes']}")

print(f"\n💡 Recommendations:")
for rec in sample_plan['recommendations']:
    print(f"  {rec}")

print("\n" + "=" * 60)

---

## Step 5: Optimization Scenarios

**Problem:** 197 invitations is a lot! Can we reduce this by improving the survey?

Let's explore optimization scenarios:

In [ ]:
# Generate optimization scenarios
optimization = optimize_survey_design(
    power_result=power_result,
    fatigue_score=fatigue_score,
    response_rate=1.0
)

print("⚡ OPTIMIZATION ANALYSIS")
print("=" * 60)

print(f"\n📊 Current State:")
print(f"  Fatigue score: {optimization['current_state']['fatigue_score']:.0f}/100")
print(f"  Risk level: {optimization['current_state']['risk_level']}")
print(f"  Drop-off rate: {optimization['current_state']['dropoff_rate']:.1%}")
print(f"  Invitations needed: {optimization['current_state']['invitations_needed']}")

if optimization['needs_optimization']:
    print(f"\n🚨 Issues:")
    for issue in optimization['issues']:
        print(f"  - {issue}")

    print(f"\n💡 Optimization Scenarios:\n")

    for i, scenario in enumerate(optimization['optimization_scenarios'], 1):
        print(f"Scenario {i}: {scenario['name']}")
        print(f"  Target fatigue score: {scenario['target_fatigue_score']:.0f}/100")
        print(f"  New drop-off rate: {scenario['new_dropoff_rate']:.1%}")
        print(f"  New invitations needed: {scenario['new_invitations']}")
        print(f"  💰 Invitations saved: {scenario['invitations_saved']}")
        print(f"  Effort: {scenario['effort']}")
        print(f"\n  Actions to take:")
        for action in scenario['actions']:
            print(f"    - {action}")
        print()

---

## Step 6: Compare Before/After

Let's see the impact of implementing **Scenario 1** (reduce to moderate risk):

In [ ]:
# Original design (HIGH risk)
original_invitations = invitation_plan['invitations_needed']

# Improved design (MODERATE risk)
improved_fatigue_score = 50
improved_dropoff = estimate_dropoff_from_score(improved_fatigue_score)
improved_plan = calculate_invitations(required_completes, improved_dropoff)

# Comparison
print("📊 BEFORE vs AFTER")
print("=" * 60)
print(f"\nORIGINAL DESIGN (High Risk):")
print(f"  Fatigue score: {fatigue_score}/100")
print(f"  Drop-off: {dropoff_rate:.1%}")
print(f"  Invitations needed: {original_invitations}")

print(f"\nIMPROVED DESIGN (Moderate Risk):")
print(f"  Fatigue score: {improved_fatigue_score}/100")
print(f"  Drop-off: {improved_dropoff:.1%}")
print(f"  Invitations needed: {improved_plan['invitations_needed']}")

print(f"\n💰 SAVINGS:")
savings = original_invitations - improved_plan['invitations_needed']
savings_pct = savings / original_invitations * 100
print(f"  {savings} fewer invitations ({savings_pct:.0f}% reduction)")
print(f"\n✅ By reducing fatigue from {fatigue_score} to {improved_fatigue_score},")
print(f"   you save {savings} invitations while maintaining {required_completes} completes!")

---

## Example 2: Segment Analysis with Optimization

**Research Question:** Do satisfaction scores differ across 3 user segments (40%, 40%, 20%)?

Let's run the full workflow for a segment study:

In [ ]:
# Step 1: Power analysis for segments
seg_calc = SegmentPowerCalc(
    n_segments=3,
    prevalence=[0.40, 0.40, 0.20],
    test_type='anova'
)

seg_power_result = seg_calc.calculate(
    effect_size=0.25,  # Cohen's f
    alpha=0.05,
    power=0.80,
    solve_for='n'
)

print("Step 1: Power Analysis")
print(f"  Total required: {seg_power_result['total_n']}")
print(f"  Per segment: {seg_power_result['n_per_segment']}")

# Step 2: Assume moderate fatigue
seg_fatigue_score = 45
seg_dropoff = estimate_dropoff_from_score(seg_fatigue_score)

print(f"\nStep 2: Fatigue Audit")
print(f"  Fatigue score: {seg_fatigue_score}/100")
print(f"  Estimated drop-off: {seg_dropoff:.1%}")

# Step 3: Calculate invitations
seg_invitations = calculate_invitations(
    required_completes=seg_power_result['total_n'],
    dropoff_rate=seg_dropoff
)

print(f"\nStep 3: Invitations Needed")
print(f"  {seg_invitations['invitations_needed']} invitations")
print(f"  (to get {seg_power_result['total_n']} completes)")

# Step 4: Show segment allocation
print(f"\nExpected completes per segment:")
for i, (n, p) in enumerate(zip(seg_power_result['n_per_segment'], [0.40, 0.40, 0.20])):
    print(f"  Segment {i+1} ({p:.0%}): {n} completes")

---

## Key Takeaways

1. **Power analysis alone underestimates needs** - You need more invitations than just the required completes

2. **Survey design impacts sample size** - High fatigue → more drop-off → more invitations

3. **Optimization is worth it** - Reducing fatigue from HIGH to MODERATE can save 20-30% of invitations

4. **Iterate before launching** - Run this workflow during design, not after fielding

## Next Steps

1. Design your survey
2. Run fatigue audit (`02-fatigue-analysis.ipynb`)
3. Come back to this notebook to calculate final invitation count
4. If invitations too high, optimize survey and re-audit
5. Repeat until satisfactory